# MeanReversion — Validation & Backtesting
### Load Checkpoint → Evaluate → Realistic Backtest ($100 start, 10% risk, OANDA conditions)

| OANDA Reality | Value |
|---------------|-------|
| Starting Balance | $100 |
| Risk per Trade | 10% ($10 at start) |
| Leverage | 30:1 (ESMA retail) |
| Spread | Pair-specific (OANDA avg) |
| Commission | $0 (OANDA Standard) |
| Min lot | 0.01 (1,000 units) |
| OANDA Account | `TBD` |

## Setup & Mount Drive

In [ ]:
# Uncomment for Colab:
# from google.colab import drive
# drive.mount("/content/drive")

import os, sys, pathlib, time, json, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.amp import autocast
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths — adjust for Colab vs local
DRIVE_ROOT = Path("/content/drive/MyDrive/phase_lm") if Path("/content/drive").exists() else Path(".")
DRIVE_CKPT = DRIVE_ROOT / "checkpoints"
DRIVE_PROC = DRIVE_ROOT / "processed_data"

# Show available checkpoints
ckpt_base = Path("checkpoints") if Path("checkpoints").exists() else DRIVE_CKPT
ckpts = sorted(ckpt_base.glob("mean_reversion_*"), key=lambda p: p.name, reverse=True)
print(f"MeanReversion checkpoints found: {len(ckpts)}")
for c in ckpts[:5]:
    print(f"  {c.name}")

## Load Model from Checkpoint

In [ ]:
from wavetrader.mean_reversion import MeanReversion
from wavetrader.config import MeanRevConfig

CKPT_DIR = ckpts[0]  # Latest
print(f"Loading checkpoint: {CKPT_DIR.name}")

# SHA-256 verify
weights_path = CKPT_DIR / "model_weights.pt"
sha_path = CKPT_DIR / "weights.sha256"
if sha_path.exists():
    expected = sha_path.read_text().split()[0]
    h = hashlib.sha256()
    with open(weights_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    actual = h.hexdigest()
    assert expected == actual, f"Checksum MISMATCH! expected={expected}, got={actual}"
    print(f"SHA-256 verified: {actual[:16]}...")

# Load config
config_path = CKPT_DIR / "config.json"
if config_path.exists():
    with open(config_path) as f:
        cfg_dict = json.load(f)
    clean = {k: v for k, v in cfg_dict.items()
             if not k.startswith("_") and k in MeanRevConfig.__dataclass_fields__}
    config = MeanRevConfig(**clean)
else:
    config = MeanRevConfig()

# Load model
model = MeanReversion(config).to(device)
checkpoint = torch.load(str(weights_path), map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint
cleaned = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(cleaned)
model.eval()
print(f"Model loaded: {model.count_parameters():,} parameters")

## Load Test Data for All Pairs

In [ ]:
from wavetrader.data import _normalise_df

PAIRS = ["GBPJPY", "EURJPY", "USDJPY", "GBPUSD"]
PAIR_NAMES = {"GBPJPY": "GBP/JPY", "EURJPY": "EUR/JPY",
              "USDJPY": "USD/JPY", "GBPUSD": "GBP/USD"}

# OANDA typical spreads per pair (pips)
OANDA_SPREADS = {"GBPJPY": 2.5, "EURJPY": 2.0, "USDJPY": 1.4, "GBPUSD": 1.6}
PIP_VALUES    = {"GBPJPY": 6.5, "EURJPY": 6.5, "USDJPY": 6.5, "GBPUSD": 10.0}

PROCESSED_DIR = Path("processed_data") if Path("processed_data").exists() else DRIVE_PROC

test_data = {}
for pair_tag in PAIRS:
    dfs = {}
    for tf in config.timeframes:
        tf_short = tf.replace("min", "m")
        path = PROCESSED_DIR / "test" / f"{pair_tag}_{tf_short}.parquet"
        if path.exists():
            df = pd.read_parquet(path)
            df = _normalise_df(df)
            dfs[tf] = df
    if len(dfs) == len(config.timeframes):
        test_data[pair_tag] = dfs
        sizes = {tf: len(df) for tf, df in dfs.items()}
        print(f"  {pair_tag}: {sizes}")
    else:
        print(f"  {pair_tag}: MISSING (got {len(dfs)}/{len(config.timeframes)} TFs)")

print(f"\nTest pairs loaded: {len(test_data)}/{len(PAIRS)}")

## Holdout Evaluation — Accuracy, Loss, Classification Report

In [ ]:
from wavetrader.dataset import MTFForexDataset, mtf_collate_fn
from wavetrader.train_mean_reversion import MeanRevLoss
from torch.utils.data import DataLoader, ConcatDataset

criterion = MeanRevLoss().to(device)
use_amp = device.type == "cuda"

# Build test datasets
test_datasets = []
for pair_tag in test_data:
    ds = MTFForexDataset(test_data[pair_tag], config)
    test_datasets.append(ds)

combined_test = ConcatDataset(test_datasets)
test_loader = DataLoader(
    combined_test, batch_size=1024, shuffle=False,
    num_workers=min(4, os.cpu_count() or 1), collate_fn=mtf_collate_fn,
    pin_memory=(device.type == "cuda"),
)

# Collect predictions
all_preds, all_labels, all_confs = [], [], []
total_loss, n_batches = 0.0, 0

model.eval()
with torch.no_grad():
    for batch in test_loader:
        labels = batch["label"].to(device, non_blocking=True)
        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }
        with autocast("cuda", enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, labels)

        total_loss += losses["total"].item()
        n_batches += 1
        all_preds.append(out["signal_logits"].argmax(-1).cpu())
        all_labels.append(labels.cpu())
        all_confs.append(out["confidence"].squeeze(-1).cpu())

preds = torch.cat(all_preds)
labels = torch.cat(all_labels)
confs = torch.cat(all_confs)

accuracy = (preds == labels).float().mean().item()
avg_loss = total_loss / max(n_batches, 1)
print(f"Overall Accuracy: {accuracy:.4f}")
print(f"Average Loss:     {avg_loss:.4f}")

# Per-class breakdown
class_names = ["BUY", "SELL", "HOLD"]
print(f"\n{'Class':>6} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>8}")
print("-" * 50)
f1s = []
for i, name in enumerate(class_names):
    tp = ((preds == i) & (labels == i)).sum().item()
    fp = ((preds == i) & (labels != i)).sum().item()
    fn = ((preds != i) & (labels == i)).sum().item()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    support = (labels == i).sum().item()
    f1s.append(f1)
    print(f"{name:>6} {precision:10.4f} {recall:10.4f} {f1:10.4f} {support:8d}")

dir_f1 = (f1s[0] + f1s[1]) / 2
print(f"\nDirectional F1 (BUY+SELL avg): {dir_f1:.4f}")

## Detailed Metrics — Confusion Matrix, Calibration

In [ ]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # Confusion matrix
    cm = torch.zeros(3, 3, dtype=torch.long)
    for p, l in zip(preds, labels):
        cm[l, p] += 1

    fig_cm = go.Figure(data=go.Heatmap(
        z=cm.numpy(), x=class_names, y=class_names,
        colorscale="Blues", text=cm.numpy(), texttemplate="%{text}",
    ))
    fig_cm.update_layout(template="plotly_dark", title="Confusion Matrix",
                         xaxis_title="Predicted", yaxis_title="Actual", height=400)
    fig_cm.show()

    # Confidence distribution
    correct_mask = preds == labels
    fig_conf = make_subplots(rows=1, cols=2,
                             subplot_titles=["Confidence Distribution", "Calibration"])
    fig_conf.add_trace(go.Histogram(x=confs[correct_mask].numpy(), name="Correct",
                                    opacity=0.7, marker_color="#4ECDC4"), row=1, col=1)
    fig_conf.add_trace(go.Histogram(x=confs[~correct_mask].numpy(), name="Incorrect",
                                    opacity=0.7, marker_color="#FF6B6B"), row=1, col=1)

    # Calibration curve
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs_mean = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confs.numpy() >= lo) & (confs.numpy() < hi)
        if mask.sum() > 0:
            bin_accs.append(correct_mask[mask].float().mean().item())
            bin_confs_mean.append(confs[mask].mean().item())
    fig_conf.add_trace(go.Scatter(x=bin_confs_mean, y=bin_accs, mode="markers+lines",
                                  name="Model", marker=dict(color="#4ECDC4")), row=1, col=2)
    fig_conf.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                                  name="Perfect", line=dict(dash="dash", color="gray")), row=1, col=2)
    fig_conf.update_layout(template="plotly_dark", height=400)
    fig_conf.show()
except ImportError:
    print("plotly not installed — skipping plots")

## Per-Pair Backtest with OANDA-Realistic Config

In [ ]:
from wavetrader.backtest import run_backtest
from wavetrader.config import BacktestConfig

INITIAL_BALANCE = 100.0
RISK_PER_TRADE = 0.10

all_results = {}

for pair_tag in test_data:
    bt_config = BacktestConfig(
        initial_balance=INITIAL_BALANCE,
        risk_per_trade=RISK_PER_TRADE,
        min_confidence=0.55,
        spread_pips=OANDA_SPREADS[pair_tag],
        pip_value=PIP_VALUES[pair_tag],
        commission_per_lot=0.0,
        leverage=30.0,
        atr_halt_multiplier=3.0,
        drawdown_reduce_threshold=0.15,
    )

    results = run_backtest(model, test_data[pair_tag], config, bt_config, device)
    all_results[pair_tag] = results

    print(f"\n{'='*60}")
    print(f"{PAIR_NAMES[pair_tag]}")
    print(f"  Trades:    {results.total_trades}")
    print(f"  Win Rate:  {results.win_rate:.1%}")
    print(f"  PF:        {results.profit_factor:.2f}")
    print(f"  Final $:   ${results.final_balance:.2f}")
    print(f"  ROI:       {(results.final_balance / INITIAL_BALANCE - 1) * 100:.1f}%")
    print(f"  Max DD:    {results.max_drawdown:.1%}")

# Summary table
print(f"\n\n{'Pair':>8} {'Trades':>7} {'WinRate':>8} {'PF':>6} {'Final $':>9} {'ROI':>7} {'MaxDD':>7}")
print("-" * 60)
for pair_tag, r in all_results.items():
    roi = (r.final_balance / INITIAL_BALANCE - 1) * 100
    print(f"{PAIR_NAMES[pair_tag]:>8} {r.total_trades:7d} {r.win_rate:7.1%} "
          f"{r.profit_factor:6.2f} ${r.final_balance:8.2f} {roi:6.1f}% {r.max_drawdown:6.1%}")

## Primary Pair Deep Dive — Period Breakdowns

In [ ]:
PRIMARY_PAIR = "GBPJPY"
primary_results = all_results.get(PRIMARY_PAIR)

if primary_results and primary_results.trades:
    trades_df = pd.DataFrame([{
        "entry_time": t.entry_time,
        "exit_time": t.exit_time,
        "direction": t.direction,
        "entry_price": t.entry_price,
        "exit_price": t.exit_price,
        "pnl": t.pnl,
        "exit_reason": t.exit_reason,
    } for t in primary_results.trades])

    trades_df["entry_dt"] = pd.to_datetime(trades_df["entry_time"])
    trades_df["exit_dt"] = pd.to_datetime(trades_df["exit_time"])
    trades_df["is_winner"] = trades_df["pnl"] > 0

    # Weekly breakdown
    trades_df["week"] = trades_df["entry_dt"].dt.isocalendar().week.astype(int)
    trades_df["year"] = trades_df["entry_dt"].dt.year
    trades_df["year_week"] = trades_df["year"].astype(str) + "-W" + trades_df["week"].astype(str).str.zfill(2)

    weekly = trades_df.groupby("year_week").agg(
        net_pnl=("pnl", "sum"),
        trades=("pnl", "count"),
        win_rate=("is_winner", "mean"),
    ).reset_index()

    print(f"\n{PAIR_NAMES[PRIMARY_PAIR]} — Week-by-Week")
    print(f"{'Week':>10} {'PnL':>10} {'Trades':>7} {'WinRate':>8}")
    print("-" * 40)
    for _, row in weekly.iterrows():
        print(f"{row['year_week']:>10} ${row['net_pnl']:9.2f} {int(row['trades']):7d} {row['win_rate']:7.1%}")

    # Monthly breakdown
    trades_df["month"] = trades_df["entry_dt"].dt.to_period("M").astype(str)
    monthly = trades_df.groupby("month").agg(
        net_pnl=("pnl", "sum"),
        trades=("pnl", "count"),
        win_rate=("is_winner", "mean"),
    ).reset_index()

    print(f"\n{PAIR_NAMES[PRIMARY_PAIR]} — Monthly")
    print(f"{'Month':>10} {'PnL':>10} {'Trades':>7} {'WinRate':>8}")
    print("-" * 40)
    for _, row in monthly.iterrows():
        print(f"{row['month']:>10} ${row['net_pnl']:9.2f} {int(row['trades']):7d} {row['win_rate']:7.1%}")
else:
    print(f"No trades for {PRIMARY_PAIR}")

## Timeline Visualizations

In [ ]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    if primary_results and primary_results.trades:
        fig = make_subplots(rows=2, cols=2, subplot_titles=[
            "Equity Curve", "PnL per Trade",
            "Cumulative PnL by Month", "Win Rate Rolling (20 trades)"
        ])

        # Equity curve
        equity = [INITIAL_BALANCE] + [INITIAL_BALANCE + sum(t.pnl for t in primary_results.trades[:i+1])
                                      for i in range(len(primary_results.trades))]
        fig.add_trace(go.Scatter(y=equity, mode="lines", name="Equity",
                                 line=dict(color="#4ECDC4")), row=1, col=1)

        # PnL per trade (bar chart)
        pnls = [t.pnl for t in primary_results.trades]
        colors = ["#4ECDC4" if p > 0 else "#FF6B6B" for p in pnls]
        fig.add_trace(go.Bar(y=pnls, marker_color=colors, name="PnL"), row=1, col=2)

        # Monthly cumulative
        if not monthly.empty:
            monthly["cum_pnl"] = monthly["net_pnl"].cumsum()
            fig.add_trace(go.Bar(x=monthly["month"], y=monthly["net_pnl"],
                                 name="Monthly PnL", marker_color="#FFE66D"), row=2, col=1)

        # Rolling win rate
        if len(trades_df) >= 20:
            rolling_wr = trades_df["is_winner"].rolling(20).mean()
            fig.add_trace(go.Scatter(y=rolling_wr, mode="lines", name="WR (20)",
                                     line=dict(color="#FF6B6B")), row=2, col=2)
            fig.add_hline(y=0.5, line_dash="dash", line_color="gray", row=2, col=2)

        fig.update_layout(template="plotly_dark", height=700, showlegend=True,
                          title=f"{PAIR_NAMES[PRIMARY_PAIR]} — MeanReversion Backtest")
        fig.show()
except ImportError:
    print("plotly not installed — skipping plots")

## Session & Time-of-Day Analysis

In [ ]:
if primary_results and primary_results.trades:
    # Hour analysis
    trades_df["hour"] = trades_df["entry_dt"].dt.hour
    hourly = trades_df.groupby("hour").agg(
        net_pnl=("pnl", "sum"),
        trades=("pnl", "count"),
        win_rate=("is_winner", "mean"),
    ).reset_index()

    print(f"\n{PAIR_NAMES[PRIMARY_PAIR]} — By Hour (UTC)")
    print(f"{'Hour':>5} {'PnL':>10} {'Trades':>7} {'WinRate':>8} {'Session':>10}")
    print("-" * 45)
    for _, row in hourly.iterrows():
        h = int(row["hour"])
        if 0 <= h < 8:
            session = "Asia"
        elif 8 <= h < 16:
            session = "London"
        else:
            session = "New York"
        print(f"{h:5d} ${row['net_pnl']:9.2f} {int(row['trades']):7d} {row['win_rate']:7.1%} {session:>10}")

    # Day of week
    trades_df["dow"] = trades_df["entry_dt"].dt.dayofweek
    dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    daily = trades_df.groupby("dow").agg(
        net_pnl=("pnl", "sum"),
        trades=("pnl", "count"),
        win_rate=("is_winner", "mean"),
    ).reset_index()

    print(f"\n{PAIR_NAMES[PRIMARY_PAIR]} — By Day of Week")
    print(f"{'Day':>5} {'PnL':>10} {'Trades':>7} {'WinRate':>8}")
    print("-" * 35)
    for _, row in daily.iterrows():
        print(f"{dow_names[int(row['dow'])]:>5} ${row['net_pnl']:9.2f} {int(row['trades']):7d} {row['win_rate']:7.1%}")
else:
    print("No trades available")

## Friction Simulation

In [ ]:
if primary_results and primary_results.trades:
    np.random.seed(42)  # Reproducible

    # OANDA-specific friction parameters
    SLIPPAGE_RANGE = (0.3, 2.0)   # pips
    SPREAD_OFFHOURS = 1.5         # multiplier during off-hours
    SPREAD_NEWS_PROB = 0.05       # probability of news spike
    SPREAD_NEWS_EXTRA = 4.0       # extra pips during news
    LOT_CAP = 0.5                 # max lot size
    LATENCY_MISS_RATE = 0.02      # probability of missed fill

    theoretical_pnl = sum(t.pnl for t in primary_results.trades)
    friction_pnl = 0.0
    realistic_trades = 0

    for t in primary_results.trades:
        # Skip if latency miss
        if np.random.random() < LATENCY_MISS_RATE:
            continue

        realistic_trades += 1
        pnl = t.pnl

        # Slippage
        slip = np.random.uniform(*SLIPPAGE_RANGE)
        pnl -= slip * PIP_VALUES.get(PRIMARY_PAIR, 6.5) * 0.01  # adjust for lot size

        # Off-hours spread widening
        entry_hour = pd.to_datetime(t.entry_time).hour if hasattr(t, "entry_time") else 12
        if entry_hour < 8 or entry_hour >= 20:
            extra_spread = OANDA_SPREADS[PRIMARY_PAIR] * (SPREAD_OFFHOURS - 1)
            pnl -= extra_spread * PIP_VALUES.get(PRIMARY_PAIR, 6.5) * 0.01

        # News spike
        if np.random.random() < SPREAD_NEWS_PROB:
            pnl -= SPREAD_NEWS_EXTRA * PIP_VALUES.get(PRIMARY_PAIR, 6.5) * 0.01

        friction_pnl += pnl

    print(f"\n{'Metric':<30} {'Theoretical':>12} {'Realistic':>12}")
    print("-" * 55)
    print(f"{'Trades':<30} {len(primary_results.trades):>12d} {realistic_trades:>12d}")
    print(f"{'Total PnL':<30} ${theoretical_pnl:>11.2f} ${friction_pnl:>11.2f}")
    print(f"{'Final Balance':<30} ${INITIAL_BALANCE + theoretical_pnl:>11.2f} ${INITIAL_BALANCE + friction_pnl:>11.2f}")
    th_roi = (theoretical_pnl / INITIAL_BALANCE) * 100
    fr_roi = (friction_pnl / INITIAL_BALANCE) * 100
    print(f"{'ROI':<30} {th_roi:>11.1f}% {fr_roi:>11.1f}%")

    # Compound projections
    test_days = (trades_df["entry_dt"].max() - trades_df["entry_dt"].min()).days
    if test_days > 0:
        daily_return = (1 + friction_pnl / INITIAL_BALANCE) ** (1 / max(test_days, 1)) - 1
        annual_return = (1 + daily_return) ** 252 - 1
        print(f"\nAnnualized ROI (realistic):  {annual_return * 100:.1f}%")
        for years in [1, 2, 3, 5]:
            projected = INITIAL_BALANCE * (1 + annual_return) ** years
            print(f"  {years}Y projection: ${projected:,.2f}")
else:
    print("No trades for friction simulation")

## Equity Curve + Sample Trades

In [ ]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    if primary_results and primary_results.trades:
        fig = make_subplots(rows=1, cols=2, subplot_titles=["Equity (Linear)", "Equity (Log)"])

        equity = [INITIAL_BALANCE]
        running = INITIAL_BALANCE
        for t in primary_results.trades:
            running += t.pnl
            equity.append(running)

        fig.add_trace(go.Scatter(y=equity, mode="lines", name="Equity",
                                 line=dict(color="#4ECDC4")), row=1, col=1)
        fig.add_trace(go.Scatter(y=equity, mode="lines", name="Equity (log)",
                                 line=dict(color="#4ECDC4")), row=1, col=2)
        fig.update_yaxes(type="log", row=1, col=2)
        fig.update_layout(template="plotly_dark", height=400,
                          title="MeanReversion Equity Curve")
        fig.show()

        # Sample trades
        print(f"\nSample Trades (first 10):")
        print(f"{'Entry':>20} {'Dir':>5} {'Entry$':>10} {'Exit$':>10} {'PnL':>8} {'Reason':>12}")
        print("-" * 70)
        for t in primary_results.trades[:10]:
            print(f"{str(t.entry_time):>20} {t.direction:>5} {t.entry_price:>10.3f} "
                  f"{t.exit_price:>10.3f} ${t.pnl:>7.2f} {t.exit_reason:>12}")
except ImportError:
    print("plotly not installed — skipping")

## Save All Results

In [ ]:
RESULTS_DIR = Path("backtest_results") / CKPT_DIR.name
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if primary_results and primary_results.trades:
    # Trade log
    trades_df.to_csv(RESULTS_DIR / "trade_log.csv", index=False)

    # Equity curve
    equity_data = []
    running = INITIAL_BALANCE
    for t in primary_results.trades:
        running += t.pnl
        equity_data.append({"trade_num": len(equity_data) + 1, "balance": round(running, 2)})
    pd.DataFrame(equity_data).to_csv(RESULTS_DIR / "equity_curve.csv", index=False)

    # Period breakdowns
    if 'weekly' in dir() and not weekly.empty:
        weekly.to_csv(RESULTS_DIR / "weekly_breakdown.csv", index=False)
    if 'monthly' in dir() and not monthly.empty:
        monthly.to_csv(RESULTS_DIR / "monthly_breakdown.csv", index=False)
    if 'hourly' in dir() and not hourly.empty:
        hourly.to_csv(RESULTS_DIR / "session_breakdown.csv", index=False)
    if 'daily' in dir() and not daily.empty:
        daily.to_csv(RESULTS_DIR / "daily_breakdown.csv", index=False)

    # Per-pair summary
    summary_rows = []
    for pair_tag, r in all_results.items():
        roi = (r.final_balance / INITIAL_BALANCE - 1) * 100
        summary_rows.append({
            "pair": PAIR_NAMES[pair_tag],
            "trades": r.total_trades,
            "win_rate": round(r.win_rate, 4),
            "profit_factor": round(r.profit_factor, 2),
            "final_balance": round(r.final_balance, 2),
            "roi_pct": round(roi, 1),
            "max_drawdown": round(r.max_drawdown, 4),
        })
    pd.DataFrame(summary_rows).to_csv(RESULTS_DIR / "per_pair_summary.csv", index=False)

    # Copy to Drive
    try:
        import shutil
        drive_results = Path("/content/drive/MyDrive/phase_lm/backtest_results") / CKPT_DIR.name
        if drive_results.parent.exists():
            shutil.copytree(str(RESULTS_DIR), str(drive_results), dirs_exist_ok=True)
            print(f"Copied to Drive: {drive_results}")
    except Exception as e:
        print(f"Drive copy skipped: {e}")

    print(f"\nResults saved to: {RESULTS_DIR}")
    print(f"Files: {[f.name for f in RESULTS_DIR.iterdir()]}")
else:
    print("No results to save")